In [16]:
# Assignment 3 - Question 1: Next-Word Prediction using MLP
# Based on the provided PDF and notebooks.

# Cell 1: Combined Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import re
from collections import Counter
import requests
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from bs4 import BeautifulSoup # For Category II
import json
import itertools
import os

# Set random seed for reproducibility
torch.manual_seed(1337)
np.random.seed(1337)

In [17]:
# Cell 2: NextWordDataset Class (Used by both Categories)
#
class NextWordDataset(Dataset):
    def __init__(self, X, y):
        # Store data as tensors
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # Return both context and target as a tuple
        return self.X[idx], self.y[idx]

In [18]:
# Category I: Natural Language (The Adventures of Sherlock Holmes)
# ---

# Cell 3: Download Data (Category I)
url = "https://www.gutenberg.org/files/1661/1661-0.txt"
response = requests.get(url)
text_data = response.text

# Simple text extraction (skipping Gutenberg header/footer)
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK THE ADVENTURES OF SHERLOCK HOLMES ***"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK THE ADVENTURES OF SHERLOCK HOLMES ***"
start_index = text_data.find(start_marker) + len(start_marker)
end_index = text_data.find(end_marker)
raw_text_cat1 = text_data[start_index:end_index]

print(f"Fetched {len(raw_text_cat1)} characters of text.")


Fetched 593653 characters of text.


In [19]:
# Cell 4: Preprocessing (Category I)
def preprocess_text_cat1(text):
    # Convert to lowercase
    text = text.lower()
    # Remove special characters except full stop (as per instruction)
    text = re.sub(r'[^a-z0-9 \\\\.]', '', text)
    # Replace full stops with a spaced version to treat it as a token
    text = text.replace('.', ' . ')
    # Split into words
    words = text.split()
    return words

words_cat1 = preprocess_text_cat1(raw_text_cat1)
print(f"Total words (Category I): {len(words_cat1)}")

Total words (Category I): 106759


In [20]:
# Cell 5: Build Vocabulary (Category I)
word_counts_cat1 = Counter(words_cat1)
# Create vocabulary, adding <UNK> for unknown words
vocab_cat1 = sorted(word_counts_cat1, key=word_counts_cat1.get, reverse=True)
vocab_cat1.append('<UNK>')
word_to_ix_cat1 = {word: i for i, word in enumerate(vocab_cat1)}
ix_to_word_cat1 = {i: word for i, word in enumerate(vocab_cat1)}
vocab_size_cat1 = len(vocab_cat1)

In [21]:
# Cell 6: Report: Vocabulary (Category I)
print(f"Vocabulary size (Category I): {vocab_size_cat1}")

# 10 most frequent words
most_frequent_cat1 = word_counts_cat1.most_common(10)
print("\n10 Most Frequent Words (Category I):")
for word, count in most_frequent_cat1:
    print(f"{word}: {count}")

# 10 least frequent words
least_frequent_cat1 = word_counts_cat1.most_common()[-10:]
print("\n10 Least Frequent Words (Category I):")
for word, count in least_frequent_cat1:
    print(f"{word}: {count}")

Vocabulary size (Category I): 13866

10 Most Frequent Words (Category I):
.: 6431
the: 4897
i: 2691
and: 2679
of: 2467
to: 2450
a: 2373
in: 1634
it: 1551
that: 1550

10 Least Frequent Words (Category I):
ofvolunteer: 1
printededitions: 1
notnecessarily: 1
paperedition: 1
pg: 1
searchfacility: 1
includes: 1
gutenbergincluding: 1
tosubscribe: 1
newsletter: 1


In [22]:
# Cell 7: Create X, y pairs & Save Vocab (Category I)
import json

CONTEXT_SIZE_CAT1 = 8
X_cat1, y_cat1 = [], []

indexed_words_cat1 = [word_to_ix_cat1.get(word, word_to_ix_cat1['<UNK>']) for word in words_cat1]

for i in range(len(indexed_words_cat1) - CONTEXT_SIZE_CAT1):
    context = indexed_words_cat1[i : i + CONTEXT_SIZE_CAT1]
    target = indexed_words_cat1[i + CONTEXT_SIZE_CAT1]
    X_cat1.append(context)
    y_cat1.append(target)

print(f"Number of (X, y) pairs (with context={CONTEXT_SIZE_CAT1}): {len(X_cat1)}")

split_idx_cat1 = int(len(X_cat1) * 0.9)
X_train_data_cat1, X_val_data_cat1 = X_cat1[:split_idx_cat1], X_cat1[split_idx_cat1:]
y_train_data_cat1, y_val_data_cat1 = y_cat1[:split_idx_cat1], y_cat1[split_idx_cat1:]

train_dataset_cat1 = NextWordDataset(X_train_data_cat1, y_train_data_cat1)
val_dataset_cat1 = NextWordDataset(X_val_data_cat1, y_val_data_cat1)

train_loader_cat1 = DataLoader(train_dataset_cat1, batch_size=1024, shuffle=True)
val_loader_cat1 = DataLoader(val_dataset_cat1, batch_size=1024, shuffle=False)

# Save the vocabulary
vocab_cat1_data = {
    'word_to_ix': word_to_ix_cat1,
    'ix_to_word': ix_to_word_cat1
}
with open('vocab_cat1.json', 'w') as f:
    json.dump(vocab_cat1_data, f)

print("Category I Dataset, DataLoaders, and vocab_cat1.json created successfully.")

Number of (X, y) pairs (with context=8): 106751
Category I Dataset, DataLoaders, and vocab_cat1.json created successfully.


In [23]:
# ---
# Category II: Structured/Domain Text (Sklearn Docs)
# ---

# Cell 15: Download Data (Category II: Sklearn Docs)

def fetch_sklearn_docs(pages):
    all_text = []
    for url in pages:
        try:
            html = requests.get(url).text
            soup = BeautifulSoup(html, "html.parser")
            # Extract text from main content area for better quality
            content = soup.find('div', {'class': 'sphx-glr-content'}) or soup.find('div', {'role': 'main'})
            if content:
                text = ' '.join([p.get_text() for p in content.find_all(['p', 'pre'])])
            else:
                text = ' '.join([p.get_text() for p in soup.find_all(['p', 'pre'])])
            all_text.append(text)
        except Exception as e:
            print(f"Could not fetch {url}: {e}")
    return all_text

pages = [
    "https://scikit-learn.org/stable/user_guide.html",
    "https://scikit-learn.org/stable/modules/svm.html",
    "https://scikit-learn.org/stable/modules/tree.html",
    "https://scikit-learn.org/stable/modules/linear_model.html",
    "https://scikit-learn.org/stable/modules/clustering.html",
    "https://scikit-learn.org/stable/modules/ensemble.html"
]

print("Fetching sklearn documentation...")
docs = fetch_sklearn_docs(pages)
raw_text_cat2 = " ".join(docs)
print(f"Fetched {len(raw_text_cat2)} characters of text.")

Fetching sklearn documentation...
Fetched 244015 characters of text.


In [24]:
# Cell 16: Preprocessing (Category II)
def preprocess_text_cat2(text):
    # Convert to lowercase
    text = text.lower()
    # Remove special characters except full stop (as per your instruction)
    text = re.sub(r'[^a-z0-9 \\.]', '', text)
    # Replace full stops with a spaced version to treat it as a token
    text = text.replace('.', ' . ')
    # Split into words
    words = text.split()
    return words

words_cat2 = preprocess_text_cat2(raw_text_cat2)
print(f"Total words (Category II): {len(words_cat2)}")

Total words (Category II): 35882


In [25]:
# Cell 17: Build Vocabulary (Category II)
word_counts_cat2 = Counter(words_cat2)
# Create vocabulary, adding <UNK> for unknown words
vocab_cat2 = sorted(word_counts_cat2, key=word_counts_cat2.get, reverse=True)
vocab_cat2.append('<UNK>')
word_to_ix_cat2 = {word: i for i, word in enumerate(vocab_cat2)}
ix_to_word_cat2 = {i: word for i, word in enumerate(vocab_cat2)}
vocab_size_cat2 = len(vocab_cat2)


In [26]:
# Cell 18: Report: Vocabulary (Category II)
print(f"Vocabulary size (Category II): {vocab_size_cat2}")

# 10 most frequent words
most_frequent_cat2 = word_counts_cat2.most_common(10)
print("\n10 Most Frequent Words (Category II):")
for word, count in most_frequent_cat2:
    print(f"{word}: {count}")

# 10 least frequent words
least_frequent_cat2 = word_counts_cat2.most_common()[-10:]
print("\n10 Least Frequent Words (Category II):")
for word, count in least_frequent_cat2:
    print(f"{word}: {count}")

Vocabulary size (Category II): 6155

10 Most Frequent Words (Category II):
.: 2654
the: 2187
of: 1032
a: 739
is: 696
to: 686
and: 552
in: 465
for: 408
be: 295

10 Least Frequent Words (Category II):
nonlinearly: 1
problemusing: 1
adaboostsamme: 1
regressionwith: 1
decisiontheoretic: 1
ofonline: 1
zhu: 1
rosset: 1
drucker: 1
learninged: 1


In [27]:
# Cell 19: Create X, y pairs & Save Vocab (Category II)
import json

CONTEXT_SIZE_CAT2 = 8
X_cat2, y_cat2 = [], []

indexed_words_cat2 = [word_to_ix_cat2.get(word, word_to_ix_cat2['<UNK>']) for word in words_cat2]

for i in range(len(indexed_words_cat2) - CONTEXT_SIZE_CAT2):
    context = indexed_words_cat2[i : i + CONTEXT_SIZE_CAT2]
    target = indexed_words_cat2[i + CONTEXT_SIZE_CAT2]
    X_cat2.append(context)
    y_cat2.append(target)

print(f"Number of (X, y) pairs (with context={CONTEXT_SIZE_CAT2}): {len(X_cat2)}")

split_idx_cat2 = int(len(X_cat2) * 0.9)
X_train_data_cat2, X_val_data_cat2 = X_cat2[:split_idx_cat2], X_cat2[split_idx_cat2:]
y_train_data_cat2, y_val_data_cat2 = y_cat2[:split_idx_cat2], y_cat2[split_idx_cat2:]

train_dataset_cat2 = NextWordDataset(X_train_data_cat2, y_train_data_cat2)
val_dataset_cat2 = NextWordDataset(X_val_data_cat2, y_val_data_cat2)

train_loader_cat2 = DataLoader(train_dataset_cat2, batch_size=1024, shuffle=True)
val_loader_cat2 = DataLoader(val_dataset_cat2, batch_size=1024, shuffle=False)

# Save the vocabulary
vocab_cat2_data = {
    'word_to_ix': word_to_ix_cat2,
    'ix_to_word': ix_to_word_cat2
}
with open('vocab_cat2.json', 'w') as f:
    json.dump(vocab_cat2_data, f)

print("Category II Dataset, DataLoaders, and vocab_cat2.json created successfully.")

Number of (X, y) pairs (with context=8): 35874
Category II Dataset, DataLoaders, and vocab_cat2.json created successfully.


In [28]:
# Cell 20: Flexible Model Definition
# This class replaces the separate model definitions for Cat I and Cat II

class NextWordMLP(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_size, hidden_dim, num_layers, activation_fn):
        super(NextWordMLP, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)

        self.layer_1 = nn.Linear(context_size * embedding_dim, hidden_dim)
        self.activation1 = activation_fn
        self.dropout1 = nn.Dropout(0.5)

        self.num_layers = num_layers
        if num_layers == 2:
            self.layer_2 = nn.Linear(hidden_dim, hidden_dim)
            self.activation2 = activation_fn
            self.dropout2 = nn.Dropout(0.5)

        # Output layer
        self.layer_3 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs).view(inputs.size(0), -1)

        out = self.activation1(self.layer_1(embeds))
        out = self.dropout1(out)

        if self.num_layers == 2:
            out = self.activation2(self.layer_2(out))
            out = self.dropout2(out)

        out = self.layer_3(out)
        return out

In [29]:
# Cell 21: Master Training Loop (All 16 Models)
import itertools

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 1. Define Hyperparameter Space ---
datasets = {
    'cat1': (train_loader_cat1, val_loader_cat1, vocab_size_cat1, word_to_ix_cat1, ix_to_word_cat1, CONTEXT_SIZE_CAT1),
    'cat2': (train_loader_cat2, val_loader_cat2, vocab_size_cat2, word_to_ix_cat2, ix_to_word_cat2, CONTEXT_SIZE_CAT2)
}
embed_dims = [32, 64]
num_layers_list = [1, 2]
activations = {'relu': nn.ReLU(), 'tanh': nn.Tanh()}
HIDDEN_DIM = 1024 # As specified in the assignment
N_EPOCHS = 100 # Reduced from 500 for faster iteration over 16 models
PATIENCE = 10

# --- 2. Iterate and Train ---
for ds_name, (train_loader, val_loader, vocab_size, word_to_ix, ix_to_word, context_size) in datasets.items():
    for embed_dim in embed_dims:
        for num_layers in num_layers_list:
            for act_name, act_fn in activations.items():

                # --- Define unique model ID and paths ---
                model_id = f"{ds_name}_embed{embed_dim}_layers{num_layers}_{act_name}"
                MODEL_PATH = f'model_{model_id}.pth'
                LOSS_PLOT_PATH = f'loss_{model_id}.png'
                TSNE_PLOT_PATH = f'tsne_{model_id}.png'

                print(f"\n--- Training Model: {model_id} ---")

                # --- 3. Initialize Model, Loss, Optimizer ---
                model = NextWordMLP(vocab_size, embed_dim, context_size, HIDDEN_DIM, num_layers, act_fn)
                model.to(device)

                optimizer = optim.Adam(model.parameters(), lr=0.001)
                loss_function = nn.CrossEntropyLoss()

                train_losses = []
                val_losses = []
                best_val_loss = float('inf')
                epochs_no_improve = 0

                # --- 4. Training Loop ---
                for epoch in range(N_EPOCHS):
                    model.train()
                    total_train_loss = 0
                    for context_batch, target_batch in train_loader:
                        context_batch, target_batch = context_batch.to(device), target_batch.to(device)
                        model.zero_grad()
                        log_probs = model(context_batch)
                        loss = loss_function(log_probs, target_batch)
                        loss.backward()
                        optimizer.step()
                        total_train_loss += loss.item()

                    # Validation
                    model.eval()
                    total_val_loss = 0
                    with torch.no_grad():
                        for context_batch, target_batch in val_loader:
                            context_batch, target_batch = context_batch.to(device), target_batch.to(device)
                            log_probs = model(context_batch)
                            loss = loss_function(log_probs, target_batch)
                            total_val_loss += loss.item()

                    avg_train_loss = total_train_loss / len(train_loader)
                    avg_val_loss = total_val_loss / len(val_loader)
                    train_losses.append(avg_train_loss)
                    val_losses.append(avg_val_loss)

                    if (epoch + 1) % 10 == 0:
                        print(f"Epoch {epoch+1}/{N_EPOCHS}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

                    # Early Stopping Check
                    if avg_val_loss < best_val_loss:
                        best_val_loss = avg_val_loss
                        torch.save(model.state_dict(), MODEL_PATH)
                        epochs_no_improve = 0
                    else:
                        epochs_no_improve += 1

                    if epochs_no_improve == PATIENCE:
                        print(f"Early stopping triggered at epoch {epoch+1}")
                        break

                print(f"Training finished. Best model saved to {MODEL_PATH}")

                # --- 5. Save Loss Plot ---
                plt.figure(figsize=(10, 5))
                plt.plot(train_losses, label='Training Loss')
                plt.plot(val_losses, label='Validation Loss')
                plt.title(f'Loss Curve - {model_id}')
                plt.xlabel('Epochs')
                plt.ylabel('Loss')
                plt.legend()
                plt.savefig(LOSS_PLOT_PATH)
                plt.close() # Close plot to free memory
                print(f"Loss plot saved to {LOSS_PLOT_PATH}")

                # --- 6. Save t-SNE Plot ---
                # Load the best model weights back
                model.load_state_dict(torch.load(MODEL_PATH))
                model.eval()

                embeddings = model.embeddings.weight.data.cpu().numpy()
                n_words_to_plot = 300
                subset_indices = np.random.choice(vocab_size, n_words_to_plot, replace=False)
                subset_embeddings = embeddings[subset_indices]
                subset_words = [ix_to_word[i] for i in subset_indices]

                # Add specific words of interest
                if ds_name == 'cat1':
                    interest_words = ['holmes', 'watson', 'sherlock', 'woman', 'baker', 'street', 'said', 'man', '.']
                else: # cat2
                    interest_words = ['model', 'fit', 'predict', 'svm', 'tree', 'linear', 'cluster', 'data', '.']

                for word in interest_words:
                    if word in word_to_ix and word not in subset_words:
                        idx = word_to_ix[word]
                        subset_words.append(word)
                        subset_embeddings = np.append(subset_embeddings, [embeddings[idx]], axis=0)

                tsne = TSNE(n_components=2, random_state=1337, perplexity=15)
                embedding_2d = tsne.fit_transform(subset_embeddings)

                plt.figure(figsize=(12, 12))
                for i, word in enumerate(subset_words):
                    x, y = embedding_2d[i, :]
                    plt.scatter(x, y, c='blue', s=10)
                    if word in interest_words:
                        plt.annotate(word, (x, y), xytext=(5, 2), textcoords='offset points', ha='right', va='bottom', color='red')
                plt.title(f't-SNE Visualization - {model_id}')
                plt.savefig(TSNE_PLOT_PATH)
                plt.close() # Close plot
                print(f"t-SNE plot saved to {TSNE_PLOT_PATH}")

print("\n--- All 16 models, plots, and vocabs saved successfully. ---")

Using device: cuda

--- Training Model: cat1_embed32_layers1_relu ---
Epoch 10/100, Train Loss: 2.5631, Val Loss: 8.6033
Early stopping triggered at epoch 12
Training finished. Best model saved to model_cat1_embed32_layers1_relu.pth
Loss plot saved to loss_cat1_embed32_layers1_relu.png
t-SNE plot saved to tsne_cat1_embed32_layers1_relu.png

--- Training Model: cat1_embed32_layers1_tanh ---
Epoch 10/100, Train Loss: 2.9764, Val Loss: 8.1475
Early stopping triggered at epoch 12
Training finished. Best model saved to model_cat1_embed32_layers1_tanh.pth
Loss plot saved to loss_cat1_embed32_layers1_tanh.png
t-SNE plot saved to tsne_cat1_embed32_layers1_tanh.png

--- Training Model: cat1_embed32_layers2_relu ---
Epoch 10/100, Train Loss: 4.9708, Val Loss: 7.1061
Early stopping triggered at epoch 15
Training finished. Best model saved to model_cat1_embed32_layers2_relu.pth
Loss plot saved to loss_cat1_embed32_layers2_relu.png
t-SNE plot saved to tsne_cat1_embed32_layers2_relu.png

--- Trainin

In [30]:
# Cell 22: Install Streamlit
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 112.3 MB/s eta 0:00:00


In [31]:
# Cell 23: Streamlit App (Updated to load all 16 models)
# This cell writes its contents to 'app.py'

%%writefile app.py
import streamlit as st
import torch
import torch.nn as nn
import torch.nn.functional as F
import re
import json
import numpy as np
import os

# --- 1. Model Definition (Must be flexible) ---

class NextWordMLP(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_size, hidden_dim, num_layers, activation_fn):
        super(NextWordMLP, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)

        self.layer_1 = nn.Linear(context_size * embedding_dim, hidden_dim)
        self.activation1 = activation_fn
        self.dropout1 = nn.Dropout(0.5)

        self.num_layers = num_layers
        if num_layers == 2:
            self.layer_2 = nn.Linear(hidden_dim, hidden_dim)
            self.activation2 = activation_fn
            self.dropout2 = nn.Dropout(0.5)

        self.layer_3 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs).view(inputs.size(0), -1)
        out = self.activation1(self.layer_1(embeds))
        out = self.dropout1(out)

        if self.num_layers == 2:
            out = self.activation2(self.layer_2(out))
            out = self.dropout2(out)

        out = self.layer_3(out)
        return out

# --- 2. Helper Functions ---

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9 \\.]', '', text)
    text = text.replace('.', ' . ')
    words = text.split()
    return words

@st.cache_resource
def load_resources(model_name, activation_name, embed_dim, num_layers):
    """
    Loads the correct model and vocabulary files based on ALL user selections.
    """

    # --- 1. Determine File Paths ---
    if "Category I" in model_name or "Natural" in model_name:
        ds_name = 'cat1'
    else:
        ds_name = 'cat2'

    act_name = activation_name.lower()

    # Construct the unique model ID
    model_id = f"{ds_name}_embed{embed_dim}_layers{num_layers}_{act_name}"

    MODEL_PATH = f'model_{model_id}.pth'
    VOCAB_PATH = f'vocab_{ds_name}.json'

    # --- 2. Check if files exist ---
    if not os.path.exists(MODEL_PATH) or not os.path.exists(VOCAB_PATH):
        st.error(f"Error: Model file `{MODEL_PATH}` or vocab file `{VOCAB_PATH}` not found.")
        st.error("Please run the master training loop in your notebook to generate all 16 models.")
        return None, None, None, None, None

    # --- 3. Load Vocabulary ---
    with open(VOCAB_PATH, 'r') as f:
        vocab = json.load(f)
    word_to_ix = vocab['word_to_ix']
    ix_to_word = {int(k): v for k, v in vocab['ix_to_word'].items()}
    vocab_size = len(word_to_ix)

    # --- 4. Define Model Architecture ---
    activation_fn = nn.ReLU() if act_name == "relu" else nn.Tanh()
    CONTEXT_SIZE = 8  # Fixed from training
    HIDDEN_DIM = 1024 # Fixed from training

    model = NextWordMLP(vocab_size, embed_dim, CONTEXT_SIZE, HIDDEN_DIM, num_layers, activation_fn)

    # --- 5. Load Trained Weights ---
    model.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device('cpu')))
    model.eval()

    # Store params for display
    params = {
        "Model ID": model_id,
        "Embedding Dim": embed_dim,
        "Context Size": CONTEXT_SIZE,
        "Hidden Dim": HIDDEN_DIM,
        "Num Layers": num_layers,
        "Activation": act_name,
        "Vocabulary Size": vocab_size
    }

    return model, word_to_ix, ix_to_word, params, model_id

def generate_text(model, word_to_ix, ix_to_word, context_size, input_text, n_words, temperature, handle_unk="mask"):
    generated_words = []
    context_words = preprocess_text(input_text)

    def get_ix(word):
        if word in word_to_ix:
            return word_to_ix[word]
        elif handle_unk == "mask":
            return word_to_ix['<UNK>']
        elif handle_unk == "skip":
            return None
        else:
            return word_to_ix['<UNK>']

    for _ in range(n_words):
        context_indices_raw = [get_ix(w) for w in context_words]
        context_indices = [idx for idx in context_indices_raw if idx is not None]

        if len(context_indices) < context_size:
            pad_indices = [word_to_ix['<UNK>']] * (context_size - len(context_indices))
            input_indices = pad_indices + context_indices
        else:
            input_indices = context_indices[-context_size:]

        context_tensor = torch.tensor([input_indices], dtype=torch.long)

        with torch.no_grad():
            log_probs = model(context_tensor)
            probs = F.softmax(log_probs / temperature, dim=1)
            predicted_index = torch.multinomial(probs, 1).item()

        predicted_word = ix_to_word.get(predicted_index, '<UNK>')
        generated_words.append(predicted_word)
        context_words.append(predicted_word)

    output_text = ' '.join(generated_words)
    output_text = output_text.replace(' .', '.').strip()
    return output_text

# --- 3. Streamlit App UI ---

st.set_page_config(layout="wide")
st.title("Next-Word Deep Learning Playground")

# --- Sidebar for Controls ---
st.sidebar.header("⚙️ Model Controls")

model_choice = st.sidebar.selectbox(
    "Category",
    ("Category I (Sherlock Holmes)", "Category II (Sklearn Docs)", "Natural"),
    index=0,
    help="Choose which trained model to use. 'Natural' is mapped to 'Category I'."
)

activation_choice = st.sidebar.selectbox(
    "Activation function",
    ("relu", "tanh"),
    help="Select the activation function."
)

embedding_dim = st.sidebar.selectbox("Embedding size", (32, 64), index=1)
num_hidden_layers = st.sidebar.selectbox("No. of hidden layers", (1, 2), index=1)

temperature = st.sidebar.slider(
    "Temperature",
    min_value=0.1, max_value=2.0, value=1.60, step=0.1,
    help="Lower values are more predictable; higher values are more creative/random."
)

seed = st.sidebar.number_input("Random Seed", value=42)
torch.manual_seed(seed)
np.random.seed(seed)

unk_handling = st.sidebar.radio(
    "How to handle unknown words?",
    ("Skip word", "Mask as <UNK>", "Find closest (embedding similarity)"),
    index=1,
    help="'Find closest' is not implemented and will default to Masking."
)
unk_param = "skip" if unk_handling == "Skip word" else "mask"


# --- Load Model and Resources ---
try:
    load_model_name = "Category I (Sherlock Holmes)" if model_choice == "Natural" else model_choice

    model, word_to_ix, ix_to_word, params, model_id = load_resources(
        load_model_name,
        activation_choice,
        embedding_dim,
        num_hidden_layers
    )

    if model:
        with st.sidebar.expander("Loaded Model Parameters", expanded=True):
            st.json(params)

        # --- Main App Interface (with Tabs) ---
        tab_generator, tab_loss, tab_tsne = st.tabs(["Generator", "Loss Curves", "TSNE Embeddings"])

        with tab_generator:
            default_text = "sherlock holmes looked at" if "cat1" in model_id else "the model is fit on"
            input_text = st.text_area("Enter input context:", default_text)
            k_words = st.slider("How many next words?", 1, 100, 13)

            if st.button("Generate Next K Words"):
                if input_text.strip():
                    with st.spinner("Generating..."):
                        generated_output = generate_text(
                            model, word_to_ix, ix_to_word,
                            params["Context Size"], input_text,
                            k_words, temperature, unk_param
                        )
                    st.subheader("Generated sequence:")
                    st.markdown(f"**{input_text}** {generated_output}")
                else:
                    st.error("Please enter some starting text.")

        with tab_loss:
            st.header("Model Training Loss Curves")
            LOSS_PLOT_PATH = f'loss_{model_id}.png'
            if os.path.exists(LOSS_PLOT_PATH):
                st.image(LOSS_PLOT_PATH, caption=f"Loss Curve for {model_id}")
            else:
                st.warning(f"Could not find loss plot: `{LOSS_PLOT_PATH}`")

        with tab_tsne:
            st.header("t-SNE Embedding Visualization")
            TSNE_PLOT_PATH = f'tsne_{model_id}.png'
            if os.path.exists(TSNE_PLOT_PATH):
                st.image(TSNE_PLOT_PATH, caption=f"t-SNE Plot for {model_id}")
            else:
                st.warning(f"Could not find t-SNE plot: `{TSNE_PLOT_PATH}`")

except Exception as e:
    st.error(f"An error occurred during model loading: {e}")
    st.exception(e)
    st.info("Ensure all 16 models (e.g., `model_cat1_embed32_layers1_relu.pth`) and vocabs (`vocab_cat1.json`, `vocab_cat2.json`) are in the same folder as `app.py`.")

Writing app.py


In [32]:
# Cell 24: Install pyngrok
!pip install pyngrok

In [ ]:
# Cell 25: Run Streamlit with ngrok
from pyngrok import ngrok

# Terminate any existing tunnels
ngrok.kill()

# Start streamlit in the background
!streamlit run app.py &

# Open a tunnel to the streamlit port (default 8501)
public_url = ngrok.connect(8501)
print(f"Your Streamlit app is live at: {public_url}")




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.87.101.14:8501

